# 03 — Il confronto (Versione 4)

Tutti i modelli finiti della Versione 4 (TF-IDF con e senza titolo, ModernBERT titolo + articolo, con e senza i
criteri NewsGuard, con i loro seed) e, come **riferimenti**, quelli della Versione 2 sul solo articolo (stesso
dataset, stesso training, stessi fold, stesse 304 testate): `feature_tuned` e ModernBERT-base seed 42, 1, 2
(copiati in `../Results/riferimenti_v2/`).

Per testata: TF-IDF col voto e con la media delle probabilità, i transformer con la media. Per le configurazioni con
più seed l'**ensemble** (media delle probabilità). Intervalli al 95% con un **bootstrap per gruppo** e differenze con
il TF-IDF di riferimento (`feature_titolo_tuned` se c'è, altrimenti `v2_feature_tuned`). Metriche principali: F1
macro a 5 livelli e binario.

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

from comune import CLASSI, RESULTS, VERSIONI, metriche_di, per_testata, per_testata_media

B = 2000
modelli = {}
for d in sorted(RESULTS.glob("feature*")):
    if (d / "FATTO").exists():
        modelli[d.name] = pd.read_csv(d / "articoli.csv")
for d in sorted((RESULTS / "transformer").glob("*")):
    if (d / "FATTO").exists():
        modelli[d.name] = pd.read_csv(d / "articoli.csv")
RIF_V2 = RESULTS.parent / "riferimenti_v2"
for f in sorted(RIF_V2.glob("*.csv")):
    modelli["v2_" + f.stem] = pd.read_csv(f)
seed = {}
for n in list(modelli):
    m = re.match(r"(.+)_s\d+$", n)
    if m:
        seed.setdefault(m[1], []).append(n)
for config, nomi in seed.items():
    if len(nomi) > 1:
        parti = [modelli[n].set_index("article_id") for n in nomi]
        e = parti[0][["domain", "fold", "y_true"]].copy(); e[CLASSI] = sum(p.loc[e.index, CLASSI] for p in parti) / len(parti)
        e["y_pred"] = e[CLASSI].to_numpy().argmax(1)
        modelli[f"{config}_ensemble{len(nomi)}"] = e.reset_index()
print("modelli:", list(modelli))
T, righe = {}, {}
for n, p in modelli.items():
    for agg, fun in ([("voto", per_testata), ("media", per_testata_media)] if n.startswith(("feature", "v2_feature")) else [("media", per_testata_media)]):
        t = fun(p); T[(n, agg)] = t
        righe[(n, agg)] = {k: v for k, v in metriche_di(t.y_true, t.y_pred).items() if not isinstance(v, list)}
tutto = pd.DataFrame(righe).T; tutto.index.names = ["modello", "aggregazione"]
tutto.to_csv(RESULTS / "03_confronto_tutte_le_metriche.csv")
tutto[["n", "f1_macro", "bin_accuratezza", "accuratezza", "tolerant_1"]].round(3)

In [ ]:
rif = ("feature_titolo_tuned", "voto") if ("feature_titolo_tuned", "voto") in T else ("v2_feature_tuned", "voto")
gruppo = pd.read_csv(VERSIONI / "fold_finale.csv").set_index("domain").gruppo
dom = T[rif].index; g = gruppo.reindex(dom).to_numpy(); gruppi = np.unique(g)
pos = {x: np.where(g == x)[0] for x in gruppi}
rng = np.random.default_rng(0)
campioni = [np.concatenate([pos[x] for x in rng.choice(gruppi, len(gruppi))]) for _ in range(B)]


def due(t, idx):
    m = metriche_di(t.y_true.to_numpy()[idx], t.y_pred.to_numpy()[idx]); return m["f1_macro"], m["bin_accuratezza"]


boot = {k: np.array([due(t.reindex(dom), idx) for idx in campioni]) for k, t in T.items()}
iv = {}
for k in T:
    b = boot[k]
    r = {"F1": tutto.loc[k, "f1_macro"], "F1 2.5%": np.percentile(b[:, 0], 2.5), "F1 97.5%": np.percentile(b[:, 0], 97.5),
         "binario": tutto.loc[k, "bin_accuratezza"], "bin 2.5%": np.percentile(b[:, 1], 2.5), "bin 97.5%": np.percentile(b[:, 1], 97.5)}
    if k != rif:
        d = b - boot[rif]
        r.update({"ΔF1": r["F1"] - tutto.loc[rif, "f1_macro"], "ΔF1 2.5%": np.percentile(d[:, 0], 2.5), "ΔF1 97.5%": np.percentile(d[:, 0], 97.5),
                  "P(ΔF1>0)": (d[:, 0] > 0).mean(), "Δbin": r["binario"] - tutto.loc[rif, "bin_accuratezza"],
                  "Δbin 2.5%": np.percentile(d[:, 1], 2.5), "Δbin 97.5%": np.percentile(d[:, 1], 97.5)})
    iv[f"{k[0]} ({k[1]})"] = r
iv = pd.DataFrame(iv).T; iv.to_csv(RESULTS / "03_confronto_intervalli.csv")
print("riferimento:", rif)
iv.round(3)